In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [2]:
PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR = PROJECT_DIR / 'data'
SRC_DIR = PROJECT_DIR / 'src'
EXTRACT_DIR = PROJECT_DIR / 'processed' / 'extraction'
OUTPUT_DIR = PROJECT_DIR / 'processed' / 'transform'
REPORT_DIR = PROJECT_DIR / 'reports'

print('PROJECT_DIR:', PROJECT_DIR)
print('EXTRACT_DIR:', EXTRACT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

PROJECT_DIR: D:\ONEASH_local\Delirium-Prediction\Parkinson
EXTRACT_DIR: D:\ONEASH_local\Delirium-Prediction\Parkinson\processed\extraction
OUTPUT_DIR: D:\ONEASH_local\Delirium-Prediction\Parkinson\processed\transform


## 데이터 로드

In [3]:
# all events (chart + lab + eMAR medication point events)
all_events = pd.read_csv(EXTRACT_DIR / 'all_events_long.csv')

print(f"all_events shape: {all_events.shape}")
print(f"Columns: {all_events.columns.tolist()}")
all_events.head()


all_events shape: (816682, 12)
Columns: ['source_table', 'subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'label', 'feature_name', 'type', 'value', 'valuenum', 'valueuom']


,source_table,subject_id,hadm_id,stay_id,charttime,itemid,label,feature_name,type,value,valuenum,valueuom
0,chartevents,14979348,22895491.0,39414466,2139-08-31 16:00:00,225664.0,Glucose finger stick (range 70-100),glucose,numeric,87,87.0,NaN
1,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220179.0,Non Invasive Blood Pressure systolic,nibp_sbp,numeric,170,170.0,mmHg
2,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220180.0,Non Invasive Blood Pressure diastolic,nibp_dbp,numeric,80,80.0,mmHg
3,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220181.0,Non Invasive Blood Pressure mean,nibp_mbp,numeric,102,102.0,mmHg
4,chartevents,14979348,22895491.0,39414466,2139-08-30 23:51:00,220045.0,Heart Rate,heart_rate,numeric,93,93.0,bpm


In [4]:
# adm_pat_icu (icu stay info)
adm_pat_icu = pd.read_csv(EXTRACT_DIR / 'adm_pat_icu.csv')
print(f"adm_pat_icu shape: {adm_pat_icu.shape}")
print(f"Columns: {adm_pat_icu.columns.tolist()}")
adm_pat_icu.head()

adm_pat_icu shape: (974, 19)
Columns: ['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'race', 'icu_los_hours']


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,anchor_year_group,dod,admittime,dischtime,deathtime,admission_type,race,icu_los_hours
0,10018328,23786647,31269608,Neuro Stepdown,Neuro Stepdown,2154-04-24 23:03:44,2154-05-02 15:55:21,7.702512,F,83,2154,2014 - 2016,2158-12-14,2154-04-24 03:15:00,2154-05-03 14:00:00,NaN,SURGICAL SAME DAY ADMISSION,WHITE,184.860278
1,10018328,28252562,37006782,Neuro Intermediate,Neuro Intermediate,2158-02-08 22:56:08,2158-02-13 23:58:23,5.043229,F,83,2154,2014 - 2016,2158-12-14,2158-02-08 22:55:00,2158-02-18 17:30:00,NaN,OBSERVATION ADMIT,WHITE,121.037500
2,10050755,20050796,37743005,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2134-02-26 01:03:02,2134-02-26 04:27:45,0.142164,M,77,2126,2008 - 2010,2134-02-26,2134-02-25 18:41:00,2134-02-26 01:26:00,2134-02-26 01:26:00,OBSERVATION ADMIT,WHITE - RUSSIAN,3.411944
3,10052769,22087051,30336368,Neuro Surgical Intensive Care Unit (Neuro SICU),Neuro Surgical Intensive Care Unit (Neuro SICU),2124-06-16 20:18:49,2124-07-08 14:31:39,21.758912,M,78,2124,2020 - 2022,2124-07-09,2124-04-25 19:33:00,2124-07-09 09:43:00,2124-07-09 09:43:00,URGENT,UNKNOWN,522.213889
4,10052769,22087051,34171709,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2124-05-25 15:48:47,2124-05-26 02:14:53,0.434792,M,78,2124,2020 - 2022,2124-07-09,2124-04-25 19:33:00,2124-07-09 09:43:00,2124-07-09 09:43:00,URGENT,UNKNOWN,10.435000


In [5]:
# Date columns
all_events['charttime'] = pd.to_datetime(all_events['charttime'], errors='coerce')
for col in ['intime', 'outtime', 'admittime', 'dischtime', 'deathtime']:
    adm_pat_icu[col] = pd.to_datetime(adm_pat_icu[col], errors='coerce')

## VALUE 변환 (문자열 → 숫자)

In [6]:
all_events.rename(columns={'value': 'value_str',
                           'valuenum': 'value'}, inplace=True)

In [7]:
# 문자열/범주형 -> 숫자 변환

# Delirium assessment: Positive=1, Negative=0, UTA/other=NaN
all_events.loc[all_events['value_str'].astype('string').str.lower().eq('positive'), 'value'] = 1
all_events.loc[all_events['value_str'].astype('string').str.lower().eq('negative'), 'value'] = 0

# RASS text values
all_events.loc[all_events['value_str'] == '"+4 Combative' , 'value']        = 4
all_events.loc[all_events['value_str'] == '+3 Pulls or removes tube(s) or catheter(s); aggressive', 'value'] = 3
all_events.loc[all_events['value_str'] == '"+2 Frequent nonpurposeful movement', 'value']                    = 2
all_events.loc[all_events['value_str'] == '"+1 Anxious', 'value']           = 1
all_events.loc[all_events['value_str'] == ' 0  Alert and calm', 'value']    = 0
all_events.loc[all_events['value_str'] == '-1 Awakens to voice (eye opening/contact) > 10 sec', 'value'] = -1
all_events.loc[all_events['value_str'] == '"-2 Light sedation' , 'value']   = -2
all_events.loc[all_events['value_str'] == '"-3 Moderate sedation', 'value'] = -3
all_events.loc[all_events['value_str'] == '"-4 Deep sedation', 'value']     = -4
all_events.loc[all_events['value_str'] == '"-5 Unarousable' , 'value']      = -5


## 단위 변환

In [8]:
if 'valueuom_original' not in all_events.columns:
    all_events['valueuom_original'] = all_events['valueuom']

def fahr_to_celsius(temp_fahr):
    return (temp_fahr - 32) * 5 / 9

# Temperature: Fahrenheit -> Celsius
temp_f_mask = all_events['label'].eq('Temperature Fahrenheit')
all_events.loc[temp_f_mask, 'value'] = fahr_to_celsius(all_events.loc[temp_f_mask, 'value'])
all_events.loc[all_events['feature_name'].eq('temperature'), 'valueuom'] = 'C'

# Weight: lbs -> kg
weight_lbs_mask = all_events['label'].eq('Admission Weight (lbs.)')
all_events.loc[weight_lbs_mask, 'value'] = all_events.loc[weight_lbs_mask, 'value'] * 0.453592
all_events.loc[all_events['feature_name'].eq('weight'), 'valueuom'] = 'kg'

# Height: inch -> cm
height_inch_mask = all_events['label'].eq('Height')
all_events.loc[height_inch_mask, 'value'] = all_events.loc[height_inch_mask, 'value'] * 2.54
all_events.loc[all_events['feature_name'].eq('height'), 'valueuom'] = 'cm'


In [9]:
all_events.to_csv(OUTPUT_DIR / 'all_events_filtered.csv', index=False)


## Delirium assessment 간격 확인

In [10]:
# 같은 ICU stay 안에서 delirium assessment가 몇 시간 간격으로 기록되는지 확인
delirium_assessment_times = (
    all_events
    .loc[
        all_events['feature_name'].eq('delirium') & all_events['charttime'].notna(),
        ['subject_id', 'hadm_id', 'stay_id', 'charttime']
    ]
    .drop_duplicates(['stay_id', 'charttime'])
    .sort_values(['stay_id', 'charttime'])
    .reset_index(drop=True)
)

delirium_assessment_times['assessment_interval_hours'] = (
    delirium_assessment_times
    .groupby('stay_id')['charttime']
    .diff()
    .dt.total_seconds()
    .div(3600)
)

delirium_interval_hours = delirium_assessment_times['assessment_interval_hours'].dropna()

delirium_interval_summary = pd.Series({
    'n_assessment_timepoints': len(delirium_assessment_times),
    'n_stays_with_assessment': delirium_assessment_times['stay_id'].nunique(),
    'n_intervals': len(delirium_interval_hours),
    'mean_interval_hours': delirium_interval_hours.mean(),
    'median_interval_hours': delirium_interval_hours.median(),
})
stay_level_interval_summary = (
    delirium_assessment_times
    .groupby('stay_id')['assessment_interval_hours']
    .agg(
        n_intervals='count',
        mean_interval_hours='mean',
        median_interval_hours='median',
    )
    .query('n_intervals > 0')
    .reset_index()
)
display(delirium_interval_summary.to_frame('value'))

,value
n_assessment_timepoints,7287.000000
n_stays_with_assessment,885.000000
n_intervals,6402.000000
mean_interval_hours,11.213915
median_interval_hours,10.100000


## 시간 계산 (ICU 입실 기준)

In [11]:
# length of stay in icu (hours)
adm_pat_icu['los_hours'] = ((adm_pat_icu['outtime'] - adm_pat_icu['intime']).dt.total_seconds() / 3600).astype(int)

In [12]:
patient_cols = [
    'stay_id', 'subject_id', 'hadm_id', 'anchor_age', 'gender', 'los_hours',
    'admission_type', 'race', 'hospital_expire_flag', 'intime', 'outtime'
]
patient_cols = [c for c in patient_cols if c in adm_pat_icu.columns]
merge_keys = [c for c in ['stay_id', 'subject_id', 'hadm_id'] if c in all_events.columns and c in patient_cols]
all_events = all_events.merge(adm_pat_icu[patient_cols], on=merge_keys, how='inner')

all_events

,source_table,subject_id,hadm_id,stay_id,charttime,itemid,label,feature_name,type,value_str,value,valueuom,valueuom_original,anchor_age,gender,los_hours,admission_type,race,intime,outtime
0,chartevents,14979348,22895491.0,39414466,2139-08-31 16:00:00,225664.0,Glucose finger stick (range 70-100),glucose,numeric,87,87.0,NaN,NaN,86,M,46,EW EMER.,WHITE,2139-08-30 23:48:00,2139-09-01 22:42:54
1,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220179.0,Non Invasive Blood Pressure systolic,nibp_sbp,numeric,170,170.0,mmHg,mmHg,86,M,46,EW EMER.,WHITE,2139-08-30 23:48:00,2139-09-01 22:42:54
2,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220180.0,Non Invasive Blood Pressure diastolic,nibp_dbp,numeric,80,80.0,mmHg,mmHg,86,M,46,EW EMER.,WHITE,2139-08-30 23:48:00,2139-09-01 22:42:54
3,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220181.0,Non Invasive Blood Pressure mean,nibp_mbp,numeric,102,102.0,mmHg,mmHg,86,M,46,EW EMER.,WHITE,2139-08-30 23:48:00,2139-09-01 22:42:54
4,chartevents,14979348,22895491.0,39414466,2139-08-30 23:51:00,220045.0,Heart Rate,heart_rate,numeric,93,93.0,bpm,bpm,86,M,46,EW EMER.,WHITE,2139-08-30 23:48:00,2139-09-01 22:42:54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
816677,emar,14240547,25871919.0,30519369,2209-11-21 14:27:00,NaN,Carbidopa-Levodopa (25-100),levodopa_related,binary,1,1.0,NaN,NaN,75,M,51,OBSERVATION ADMIT,WHITE,2209-11-20 17:36:23,2209-11-22 21:28:22
816678,emar,14240547,25871919.0,30519369,2209-11-21 20:32:00,NaN,Carbidopa-Levodopa (25-100),levodopa_related,binary,1,1.0,NaN,NaN,75,M,51,OBSERVATION ADMIT,WHITE,2209-11-20 17:36:23,2209-11-22 21:28:22
816679,emar,14240547,25871919.0,30519369,2209-11-22 08:03:00,NaN,Carbidopa-Levodopa (25-100),levodopa_related,binary,1,1.0,NaN,NaN,75,M,51,OBSERVATION ADMIT,WHITE,2209-11-20 17:36:23,2209-11-22 21:28:22
816680,emar,14240547,25871919.0,30519369,2209-11-22 13:50:00,NaN,Carbidopa-Levodopa (25-100),levodopa_related,binary,1,1.0,NaN,NaN,75,M,51,OBSERVATION ADMIT,WHITE,2209-11-20 17:36:23,2209-11-22 21:28:22


## 12시간 구간 라벨링 (long-format events)

In [13]:
# 12시간 단위 라벨링
all_events['_elapsed_hours'] = (
    (all_events['charttime'] - all_events['intime']).dt.total_seconds() / 3600.0
)

all_events = all_events.loc[
    (all_events['_elapsed_hours'] >= 0) &
    (all_events['charttime'] <= all_events['outtime'])
].copy()

all_events['bin'] = np.ceil((all_events['_elapsed_hours'] - 1e-9) / 12).clip(lower=1).astype('int64')
all_events['hours'] = all_events['bin'] * 12
all_events.drop(columns='_elapsed_hours', inplace=True)

# event row는 집계/pivot 없이 시간순으로 유지
sort_cols = ['stay_id', 'charttime', 'bin', 'feature_name', 'itemid']
timeseries = all_events.sort_values(sort_cols).reset_index(drop=True)

preview_cols = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'bin', 'hours', 'feature_name', 'value', 'valueuom']

print("\n=== Long-format events with 12-hour labels ===")
print(f"Unique stay_ids: {timeseries['stay_id'].nunique()}")
print(f"Total event rows: {len(timeseries):,}")
display(timeseries[preview_cols].head(20))
timeseries.to_csv(OUTPUT_DIR / 'all_events_12h_long.csv', index=False)
print(f"Saved: all_events_12h_long.csv ({len(timeseries):,} rows, {timeseries['stay_id'].nunique()} stays)")


=== Long-format events with 12-hour labels ===
Unique stay_ids: 974
Total event rows: 816,682


,subject_id,hadm_id,stay_id,charttime,bin,hours,feature_name,value,valueuom
0,19448158,28451027.0,30008046,2154-01-26 17:57:00,1,12,heart_rate,87.000000,bpm
1,19448158,28451027.0,30008046,2154-01-26 17:57:00,1,12,nibp_dbp,74.000000,mmHg
2,19448158,28451027.0,30008046,2154-01-26 17:57:00,1,12,nibp_mbp,92.000000,mmHg
3,19448158,28451027.0,30008046,2154-01-26 17:57:00,1,12,nibp_sbp,134.000000,mmHg
4,19448158,28451027.0,30008046,2154-01-26 17:57:00,1,12,respiratory_rate,27.000000,insp/min
5,19448158,28451027.0,30008046,2154-01-26 17:57:00,1,12,spo2,95.000000,%
6,19448158,28451027.0,30008046,2154-01-26 18:00:00,1,12,delirium,0.000000,NaN
7,19448158,28451027.0,30008046,2154-01-26 18:00:00,1,12,gcs_eye,4.000000,NaN
8,19448158,28451027.0,30008046,2154-01-26 18:00:00,1,12,gcs_motor,5.000000,NaN
9,19448158,28451027.0,30008046,2154-01-26 18:00:00,1,12,gcs_verbal,1.000000,NaN


Saved: all_events_12h_long.csv (816,682 rows, 974 stays)


## 12시간 구간 라벨링 (wide-format by charttime)

In [14]:
wide_index_cols = [c for c in [
    'subject_id', 'hadm_id', 'stay_id', 'charttime', 'bin', 'hours',
    'age', 'gender', 'los_hours', 'admission_type', 'race', 'hospital_expire_flag',
    'intime', 'outtime'
] if c in timeseries.columns]

wide_source = (
    timeseries
    .loc[timeseries['feature_name'].notna(), wide_index_cols + ['feature_name', 'value']]
    .sort_values([c for c in ['stay_id', 'charttime', 'bin', 'feature_name'] if c in timeseries.columns])
)

duplicate_key_cols = wide_index_cols + ['feature_name']
duplicate_feature_rows = wide_source.duplicated(duplicate_key_cols, keep=False).sum()
if duplicate_feature_rows:
    print(f"Same stay/charttime/feature duplicate rows: {duplicate_feature_rows:,}; keeping the first observed value for exact duplicates only.")
    wide_source = wide_source.drop_duplicates(duplicate_key_cols, keep='first')

wide_timeseries = (
    wide_source
    .pivot(index=wide_index_cols, columns='feature_name', values='value')
    .reset_index()
    .sort_values([c for c in ['stay_id', 'charttime', 'bin'] if c in wide_index_cols])
    .reset_index(drop=True)
)
wide_timeseries.columns.name = None

# height/weight는 가장 처음 측정된 값 사용
static_fill_cols = ['height', 'weight']
static_fill_cols = [col for col in static_fill_cols if col in wide_timeseries.columns]

for col in static_fill_cols:
    first_values = (
        wide_timeseries
        .sort_values(['stay_id', 'charttime'])
        .groupby('stay_id')[col]
        .transform(lambda values: values.dropna().iloc[0] if values.notna().any() else np.nan)
    )
    wide_timeseries[col] = first_values

wide_timeseries

Same stay/charttime/feature duplicate rows: 13,194; keeping the first observed value for exact duplicates only.


,subject_id,hadm_id,stay_id,charttime,bin,hours,gender,los_hours,admission_type,race,...,rass,respiratory_rate,sao2,sedatives,sodium,spo2,temperature,vasopressor,wbc,weight
0,19448158,28451027.0,30008046,2154-01-26 17:57:00,1,12,M,99,URGENT,WHITE,...,NaN,27.0,NaN,NaN,NaN,95.0,NaN,NaN,NaN,81.147609
1,19448158,28451027.0,30008046,2154-01-26 18:00:00,1,12,M,99,URGENT,WHITE,...,0.0,29.0,NaN,NaN,NaN,97.0,37.111111,NaN,NaN,81.147609
2,19448158,28451027.0,30008046,2154-01-26 18:04:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81.147609
3,19448158,28451027.0,30008046,2154-01-26 18:20:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,96.0,NaN,NaN,NaN,81.147609
4,19448158,28451027.0,30008046,2154-01-26 19:21:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81.147609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188300,14606237,27829512.0,39993425,2143-10-29 13:40:00,6,72,F,63,OBSERVATION ADMIT,WHITE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188301,14606237,27829512.0,39993425,2143-10-29 14:00:00,6,72,F,63,OBSERVATION ADMIT,WHITE,...,NaN,14.0,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN
188302,14606237,27829512.0,39993425,2143-10-29 14:01:00,6,72,F,63,OBSERVATION ADMIT,WHITE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188303,14606237,27829512.0,39993425,2143-10-29 15:00:00,6,72,F,63,OBSERVATION ADMIT,WHITE,...,NaN,15.0,NaN,NaN,NaN,99.0,NaN,NaN,NaN,NaN


## 처치/장치 노출 추가

In [15]:
# procedure/device interval과 charttime이 실제로 겹치는 row만 1로 표시
procedure_events_path = EXTRACT_DIR / 'procedure_selected.csv'
procedure_events = pd.read_csv(procedure_events_path)

for col in ['starttime', 'endtime', 'intime', 'outtime']:
    if len(procedure_events) and col in procedure_events.columns:
        procedure_events[col] = pd.to_datetime(procedure_events[col], errors='coerce')

In [16]:
procedure_intervals = pd.DataFrame(columns=['stay_id', 'start_use', 'end_use', 'feature_name'])

proc = procedure_events.copy()
proc['start_use'] = proc['starttime']
proc['end_use'] = proc['endtime'].fillna(proc['starttime'])
proc = proc.loc[
    proc['feature_name'].notna()
    & proc['start_use'].notna()
    & proc['end_use'].notna()
    & proc['intime'].notna()
    & proc['outtime'].notna()
].copy()
proc['start_use'] = proc[['start_use', 'intime']].max(axis=1)
proc['end_use'] = proc[['end_use', 'outtime']].min(axis=1)
proc = proc.loc[proc['start_use'] <= proc['end_use']].copy()
procedure_intervals = (
    proc[['stay_id', 'start_use', 'end_use', 'feature_name']]
    .drop_duplicates()
    .sort_values(['stay_id', 'start_use', 'end_use', 'feature_name'])
    .reset_index(drop=True)
)

procedure_binary_cols = sorted(procedure_intervals['feature_name'].dropna().unique().tolist())
for col in procedure_binary_cols:
    wide_timeseries[col] = 0

for stay_id, stay_proc in procedure_intervals.groupby('stay_id'):
    stay_idx = wide_timeseries.index[wide_timeseries['stay_id'].eq(stay_id)]
    if len(stay_idx) == 0:
        continue
    stay_charttime = wide_timeseries.loc[stay_idx, 'charttime']
    for row in stay_proc.itertuples(index=False):
        exposed_idx = stay_charttime.index[
            (stay_charttime >= row.start_use) & (stay_charttime <= row.end_use)
        ]
        if len(exposed_idx):
            wide_timeseries.loc[exposed_idx, row.feature_name] = 1

for col in procedure_binary_cols:
    wide_timeseries[col] = wide_timeseries[col].astype('int8')

print(f"Procedure/device columns added: {procedure_binary_cols}")
if procedure_binary_cols:
    display(wide_timeseries.loc[wide_timeseries[procedure_binary_cols].sum(axis=1) > 0, ['stay_id', 'charttime', 'bin', 'hours'] + procedure_binary_cols].head(10))


Procedure/device columns added: ['procedure_invasive_ventilation', 'procedure_noninvasive_ventilation']


,stay_id,charttime,bin,hours,procedure_invasive_ventilation,procedure_noninvasive_ventilation
727,30054857,2185-08-03 23:31:00,1,12,1,0
728,30054857,2185-08-03 23:38:00,1,12,1,0
729,30054857,2185-08-03 23:47:00,1,12,1,0
730,30054857,2185-08-04 00:00:00,1,12,1,0
731,30054857,2185-08-04 00:01:00,1,12,1,0
732,30054857,2185-08-04 00:50:00,1,12,1,0
733,30054857,2185-08-04 01:00:00,1,12,1,0
734,30054857,2185-08-04 01:01:00,1,12,1,0
735,30054857,2185-08-04 01:31:00,1,12,1,0
736,30054857,2185-08-04 02:00:00,1,12,1,0


## delirium 라벨 생성


In [17]:
# 12시간 window 단위 delirium label 생성
# 해당 bin 안 assessment 중 하나라도 positive면 1, assessment는 있지만 positive가 없으면 0, assessment가 없으면 null.
if 'delirium' in wide_timeseries.columns:
    wide_timeseries.drop(columns='delirium', inplace=True)

delirium_12h = (
    timeseries
    .loc[timeseries['feature_name'].eq('delirium') & timeseries['value'].notna(), ['stay_id', 'bin', 'value']]
    .groupby(['stay_id', 'bin'], as_index=False)['value']
    .max()
    .rename(columns={'value': 'delirium'})
)
delirium_12h['delirium'] = delirium_12h['delirium'].astype('int8')

wide_timeseries = wide_timeseries.merge(delirium_12h, on=['stay_id', 'bin'], how='left')
wide_timeseries['delirium'] = wide_timeseries['delirium'].astype('Int8')

print('12-hour delirium label 분포:')
print(delirium_12h['delirium'].value_counts())
display(wide_timeseries.head(20))

12-hour delirium label 분포:
delirium
0    2778
1    1570
Name: count, dtype: int64


,subject_id,hadm_id,stay_id,charttime,bin,hours,gender,los_hours,admission_type,race,...,sedatives,sodium,spo2,temperature,vasopressor,wbc,weight,procedure_invasive_ventilation,procedure_noninvasive_ventilation,delirium
0,19448158,28451027.0,30008046,2154-01-26 17:57:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,95.0,NaN,NaN,NaN,81.147609,0,0,0
1,19448158,28451027.0,30008046,2154-01-26 18:00:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,97.0,37.111111,NaN,NaN,81.147609,0,0,0
2,19448158,28451027.0,30008046,2154-01-26 18:04:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,NaN,81.147609,0,0,0
3,19448158,28451027.0,30008046,2154-01-26 18:20:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,96.0,NaN,NaN,NaN,81.147609,0,0,0
4,19448158,28451027.0,30008046,2154-01-26 19:21:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,NaN,81.147609,0,0,0
5,19448158,28451027.0,30008046,2154-01-26 20:00:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,97.0,36.666667,NaN,NaN,81.147609,0,0,0
6,19448158,28451027.0,30008046,2154-01-26 20:01:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,NaN,81.147609,0,0,0
7,19448158,28451027.0,30008046,2154-01-26 20:04:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,NaN,81.147609,0,0,0
8,19448158,28451027.0,30008046,2154-01-26 22:00:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,100.0,NaN,NaN,NaN,81.147609,0,0,0
9,19448158,28451027.0,30008046,2154-01-26 22:04:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,NaN,81.147609,0,0,0


In [18]:
# stay_id 단위로 delirium이 ever positive였는지 여부 계산

subject_ever_delirium = (
    wide_timeseries
    .dropna(subset=['delirium'])
    .groupby('subject_id')['delirium']
    .max()
    .reindex(wide_timeseries['subject_id'].dropna().unique(), fill_value=0)
    .fillna(0)
    .astype('int8')
)

wide_timeseries['ever_delirium'] = wide_timeseries['subject_id'].map(subject_ever_delirium).fillna(0).astype('int8')


print('ever_delirium subject 분포:')
print(subject_ever_delirium.value_counts().sort_index())
display(wide_timeseries.head(50))

wide_timeseries.to_csv(OUTPUT_DIR / 'all_events_12h_wide_by_charttime.csv', index=False)
print(f"Saved: all_events_12h_wide_by_charttime.csv ({len(wide_timeseries):,} rows, {wide_timeseries['stay_id'].nunique()} stays)")

ever_delirium subject 분포:
delirium
0    278
1    290
Name: count, dtype: int64


,subject_id,hadm_id,stay_id,charttime,bin,hours,gender,los_hours,admission_type,race,...,sodium,spo2,temperature,vasopressor,wbc,weight,procedure_invasive_ventilation,procedure_noninvasive_ventilation,delirium,ever_delirium
0,19448158,28451027.0,30008046,2154-01-26 17:57:00,1,12,M,99,URGENT,WHITE,...,NaN,95.0,NaN,NaN,NaN,81.147609,0,0,0,0
1,19448158,28451027.0,30008046,2154-01-26 18:00:00,1,12,M,99,URGENT,WHITE,...,NaN,97.0,37.111111,NaN,NaN,81.147609,0,0,0,0
2,19448158,28451027.0,30008046,2154-01-26 18:04:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,81.147609,0,0,0,0
3,19448158,28451027.0,30008046,2154-01-26 18:20:00,1,12,M,99,URGENT,WHITE,...,NaN,96.0,NaN,NaN,NaN,81.147609,0,0,0,0
4,19448158,28451027.0,30008046,2154-01-26 19:21:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,81.147609,0,0,0,0
5,19448158,28451027.0,30008046,2154-01-26 20:00:00,1,12,M,99,URGENT,WHITE,...,NaN,97.0,36.666667,NaN,NaN,81.147609,0,0,0,0
6,19448158,28451027.0,30008046,2154-01-26 20:01:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,81.147609,0,0,0,0
7,19448158,28451027.0,30008046,2154-01-26 20:04:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,81.147609,0,0,0,0
8,19448158,28451027.0,30008046,2154-01-26 22:00:00,1,12,M,99,URGENT,WHITE,...,NaN,100.0,NaN,NaN,NaN,81.147609,0,0,0,0
9,19448158,28451027.0,30008046,2154-01-26 22:04:00,1,12,M,99,URGENT,WHITE,...,NaN,NaN,NaN,NaN,NaN,81.147609,0,0,0,0


Saved: all_events_12h_wide_by_charttime.csv (188,305 rows, 974 stays)


## 12시간 binning

변수별 측정 빈도 확인

In [22]:
# Median measurement interval by variable for chart/lab events
chart_lab_times = (
    timeseries
    .loc[
        timeseries['source_table'].isin(['chartevents', 'labevents'])
        & timeseries['feature_name'].notna()
        & timeseries['charttime'].notna()
        & timeseries['value'].notna(),
        ['source_table', 'feature_name', 'subject_id', 'charttime']
    ]
    .drop_duplicates(subset=['source_table', 'feature_name', 'subject_id', 'charttime'])
    .sort_values(['source_table', 'feature_name', 'subject_id', 'charttime'])
    .copy()
)

chart_lab_times['interval_hours'] = (
    chart_lab_times
    .groupby(['source_table', 'feature_name', 'subject_id'])['charttime']
    .diff()
    .dt.total_seconds() / 3600
)

chart_lab_interval_medians = (
    chart_lab_times
    .dropna(subset=['interval_hours'])
    .groupby(['source_table', 'feature_name'], as_index=False)['interval_hours']
    .median()
    .rename(columns={'interval_hours': 'median_interval_hours'})
    .sort_values(['source_table', 'feature_name'])
)

display(chart_lab_interval_medians)


,source_table,feature_name,median_interval_hours
0,chartevents,abp_dbp,1.000000
1,chartevents,abp_mbp,1.000000
2,chartevents,abp_sbp,1.000000
3,chartevents,delirium,11.766667
4,chartevents,gcs_eye,4.000000
5,chartevents,gcs_motor,4.000000
6,chartevents,gcs_verbal,4.000000
7,chartevents,glucose,4.466667
8,chartevents,heart_rate,1.000000
9,chartevents,nibp_dbp,1.000000


빈번한 V/S, glucose 등은 요약정보 사용.

lab 결과는 bin 내에서 가장 최근 값 사용. 값이 없다면 직전 bin에서 측정된 값으로 imputation. (그럼에도 불구하고 결측치 많은 변수는 추후 train set에서 확인 후 제거 예정)

In [26]:
# 12-hour bin-level wide table using extraction_variable_catalog.csv
# - binning == 'aggregation': mean/median/std/count/min/max/latest within the bin
# - binning == 'most recent': latest value in the bin, then carry-forward from previous bins
# - binning == 'at least once': 1 if present/exposed at least once within the bin
# - binning == 'static': baseline/static values repeated across bins

variable_catalog = pd.read_csv(SRC_DIR / 'extraction_variable_catalog.csv')
variable_catalog['binning'] = variable_catalog['binning'].str.strip().str.lower()

feature_binning = (
    variable_catalog
    .dropna(subset=['feature_name', 'binning'])
    [['feature_name', 'binning']]
    .drop_duplicates()
)

procedure_catalog_features = set(
    variable_catalog.loc[variable_catalog['source_table'].eq('procedureevents'), 'feature_name'].dropna()
)
aggregation_features = sorted(feature_binning.loc[feature_binning['binning'].eq('aggregation'), 'feature_name'].unique())
most_recent_features = sorted(feature_binning.loc[feature_binning['binning'].eq('most recent'), 'feature_name'].unique())
static_features_from_events = sorted(
    set(feature_binning.loc[feature_binning['binning'].eq('static'), 'feature_name'])
    - {'age', 'gender'}
)
at_least_once_features = sorted(
    set(feature_binning.loc[feature_binning['binning'].eq('at least once'), 'feature_name'])
    - procedure_catalog_features
)
procedure_features = sorted(
    variable_catalog.loc[
        variable_catalog['source_table'].eq('procedureevents')
        & variable_catalog['binning'].eq('at least once'),
        'feature_name'
    ].dropna().unique()
)

adm_for_bin = adm_pat_icu.copy()
for col in ['intime', 'outtime']:
    adm_for_bin[col] = pd.to_datetime(adm_for_bin[col], errors='coerce')
adm_for_bin['los_hours'] = (adm_for_bin['outtime'] - adm_for_bin['intime']).dt.total_seconds() / 3600
adm_for_bin['total_bins'] = np.ceil(adm_for_bin['los_hours'].fillna(0).clip(lower=0) / 12).astype('int64')
if 'anchor_age' in adm_for_bin.columns and 'age' not in adm_for_bin.columns:
    adm_for_bin['age'] = adm_for_bin['anchor_age']

base_bin_rows = adm_for_bin.loc[adm_for_bin['total_bins'] > 0].copy()
base_bin_rows = base_bin_rows.loc[base_bin_rows.index.repeat(base_bin_rows['total_bins'])].copy()
base_bin_rows['bin'] = base_bin_rows.groupby('stay_id').cumcount() + 1
base_bin_rows['hours'] = base_bin_rows['bin'] * 12
base_bin_rows['bin_start'] = base_bin_rows['intime'] + pd.to_timedelta((base_bin_rows['bin'] - 1) * 12, unit='h')
base_bin_rows['bin_end'] = base_bin_rows['intime'] + pd.to_timedelta(base_bin_rows['bin'] * 12, unit='h')

base_cols = [c for c in [
    'subject_id', 'hadm_id', 'stay_id', 'bin', 'hours', 'bin_start', 'bin_end',
    'age', 'gender', 'los_hours', 'admission_type', 'race', 'hospital_expire_flag',
    'intime', 'outtime'
] if c in base_bin_rows.columns]
wide_timeseries_12h_bin = base_bin_rows[base_cols].sort_values(['stay_id', 'bin']).reset_index(drop=True)

# Static event-derived features, e.g. height/weight: first observed value per stay.
static_event_values = (
    timeseries
    .loc[
        timeseries['feature_name'].isin(static_features_from_events)
        & timeseries['charttime'].notna()
        & timeseries['value'].notna(),
        ['stay_id', 'feature_name', 'charttime', 'value']
    ]
    .sort_values(['stay_id', 'feature_name', 'charttime'])
    .drop_duplicates(['stay_id', 'feature_name'], keep='first')
)
if len(static_event_values):
    static_event_values = static_event_values.pivot(index='stay_id', columns='feature_name', values='value').reset_index()
    static_event_values.columns.name = None
    wide_timeseries_12h_bin = wide_timeseries_12h_bin.merge(static_event_values, on='stay_id', how='left')

# Aggregated features: summary statistics plus latest value within each bin.
aggregation_source = timeseries.loc[
    timeseries['feature_name'].isin(aggregation_features)
    & timeseries['charttime'].notna()
    & timeseries['value'].notna(),
    ['stay_id', 'bin', 'feature_name', 'charttime', 'value']
].copy()

aggregation_summary = (
    aggregation_source
    .groupby(['stay_id', 'bin', 'feature_name'])['value']
    .agg(['mean', 'median', 'std', 'count', 'min', 'max'])
)
aggregation_latest = (
    aggregation_source
    .sort_values(['stay_id', 'bin', 'feature_name', 'charttime'])
    .drop_duplicates(['stay_id', 'bin', 'feature_name'], keep='last')
    .set_index(['stay_id', 'bin', 'feature_name'])['value']
    .rename('latest')
)
aggregation_summary = aggregation_summary.join(aggregation_latest, how='left')

if len(aggregation_summary):
    aggregation_summary = aggregation_summary.unstack('feature_name')
    aggregation_summary.columns = [f'{feature}_{stat}' for stat, feature in aggregation_summary.columns]
    aggregation_summary = aggregation_summary.reset_index()
    wide_timeseries_12h_bin = wide_timeseries_12h_bin.merge(aggregation_summary, on=['stay_id', 'bin'], how='left')

# Most-recent features: latest result in each bin, forward-filled within stay.
most_recent_source = (
    timeseries
    .loc[
        timeseries['feature_name'].isin(most_recent_features)
        & timeseries['charttime'].notna()
        & timeseries['value'].notna(),
        ['stay_id', 'bin', 'feature_name', 'charttime', 'value']
    ]
    .sort_values(['stay_id', 'bin', 'feature_name', 'charttime'])
    .drop_duplicates(['stay_id', 'bin', 'feature_name'], keep='last')
)
most_recent_wide = most_recent_source.pivot(index=['stay_id', 'bin'], columns='feature_name', values='value').reset_index()
if len(most_recent_wide):
    most_recent_wide.columns.name = None
    most_recent_cols = [c for c in most_recent_wide.columns if c not in ['stay_id', 'bin']]
    wide_timeseries_12h_bin = wide_timeseries_12h_bin.merge(most_recent_wide, on=['stay_id', 'bin'], how='left')
    wide_timeseries_12h_bin[most_recent_cols] = (
        wide_timeseries_12h_bin
        .sort_values(['stay_id', 'bin'])
        .groupby('stay_id')[most_recent_cols]
        .ffill()
    )

# At-least-once point features, e.g. medication and delirium assessment.
at_least_once_source = timeseries.loc[
    timeseries['feature_name'].isin(at_least_once_features)
    & timeseries['value'].notna(),
    ['stay_id', 'bin', 'feature_name', 'value']
].copy()
at_least_once_wide = (
    at_least_once_source
    .groupby(['stay_id', 'bin', 'feature_name'])['value']
    .max()
    .unstack('feature_name')
    .reset_index()
)
if len(at_least_once_wide):
    at_least_once_wide.columns.name = None
    at_least_once_cols = [c for c in at_least_once_wide.columns if c not in ['stay_id', 'bin']]
    wide_timeseries_12h_bin = wide_timeseries_12h_bin.merge(at_least_once_wide, on=['stay_id', 'bin'], how='left')
    wide_timeseries_12h_bin[at_least_once_cols] = wide_timeseries_12h_bin[at_least_once_cols].fillna(0).astype('int8')

# Procedure/device intervals: positive if the procedure interval overlaps the 12-hour bin.
procedure_events = pd.read_csv(EXTRACT_DIR / 'procedure_selected.csv')
for col in ['starttime', 'endtime', 'intime', 'outtime']:
    if col in procedure_events.columns:
        procedure_events[col] = pd.to_datetime(procedure_events[col], errors='coerce')

proc = procedure_events.loc[procedure_events['feature_name'].isin(procedure_features)].copy()
proc['start_use'] = proc['starttime']
proc['end_use'] = proc['endtime'].fillna(proc['starttime'])
proc = proc.loc[
    proc['stay_id'].notna()
    & proc['feature_name'].notna()
    & proc['start_use'].notna()
    & proc['end_use'].notna()
    & proc['intime'].notna()
    & proc['outtime'].notna()
].copy()
proc['start_use'] = proc[['start_use', 'intime']].max(axis=1)
proc['end_use'] = proc[['end_use', 'outtime']].min(axis=1)
proc = proc.loc[proc['start_use'] <= proc['end_use']].copy()
proc['start_bin'] = np.ceil(((proc['start_use'] - proc['intime']).dt.total_seconds() / 3600 - 1e-9) / 12).clip(lower=1).astype('int64')
proc['end_bin'] = np.ceil(((proc['end_use'] - proc['intime']).dt.total_seconds() / 3600 - 1e-9) / 12).clip(lower=1).astype('int64')

procedure_bin_rows = []
for row in proc[['stay_id', 'feature_name', 'start_bin', 'end_bin']].drop_duplicates().itertuples(index=False):
    for bin_id in range(row.start_bin, row.end_bin + 1):
        procedure_bin_rows.append((row.stay_id, bin_id, row.feature_name, 1))

procedure_bin = pd.DataFrame(procedure_bin_rows, columns=['stay_id', 'bin', 'feature_name', 'value'])
if len(procedure_bin):
    procedure_bin = (
        procedure_bin
        .groupby(['stay_id', 'bin', 'feature_name'])['value']
        .max()
        .unstack('feature_name')
        .reset_index()
    )
    procedure_bin.columns.name = None
    procedure_cols = [c for c in procedure_bin.columns if c not in ['stay_id', 'bin']]
    wide_timeseries_12h_bin = wide_timeseries_12h_bin.merge(procedure_bin, on=['stay_id', 'bin'], how='left')
    wide_timeseries_12h_bin[procedure_cols] = wide_timeseries_12h_bin[procedure_cols].fillna(0).astype('int8')

wide_timeseries_12h_bin = wide_timeseries_12h_bin.sort_values(['stay_id', 'bin']).reset_index(drop=True)
wide_timeseries_12h_bin.to_csv(OUTPUT_DIR / 'all_events_12h_binned.csv', index=False)

print(f"Saved: all_events_12h_binned.csv ({len(wide_timeseries_12h_bin):,} rows, {wide_timeseries_12h_bin['stay_id'].nunique()} stays)")
print('Binning rules from catalog:')
print(feature_binning['binning'].value_counts().sort_index())
display(wide_timeseries_12h_bin.head(20))


Saved: all_events_12h_binned.csv (8,008 rows, 974 stays)
Binning rules from catalog:
binning
aggregation      16
at least once    14
most recent      27
static            4
Name: count, dtype: int64


,subject_id,hadm_id,stay_id,bin,hours,bin_start,bin_end,age,gender,los_hours,...,comt_inhibitor,delirium,dopamine_agonist,levodopa_related,maob_inhibitor,opioid,sedatives,vasopressor,procedure_invasive_ventilation,procedure_noninvasive_ventilation
0,19448158,28451027,30008046,1,12,2154-01-26 17:39:49,2154-01-27 05:39:49,75,M,99.630556,...,1,0,0,1,0,0,0,0,0,0
1,19448158,28451027,30008046,2,24,2154-01-27 05:39:49,2154-01-27 17:39:49,75,M,99.630556,...,1,0,0,1,1,0,0,0,0,0
2,19448158,28451027,30008046,3,36,2154-01-27 17:39:49,2154-01-28 05:39:49,75,M,99.630556,...,1,0,0,1,0,0,0,0,0,0
3,19448158,28451027,30008046,4,48,2154-01-28 05:39:49,2154-01-28 17:39:49,75,M,99.630556,...,1,0,0,1,1,0,0,0,0,0
4,19448158,28451027,30008046,5,60,2154-01-28 17:39:49,2154-01-29 05:39:49,75,M,99.630556,...,1,0,0,1,0,0,0,0,0,0
5,19448158,28451027,30008046,6,72,2154-01-29 05:39:49,2154-01-29 17:39:49,75,M,99.630556,...,1,0,0,1,1,0,0,0,0,0
6,19448158,28451027,30008046,7,84,2154-01-29 17:39:49,2154-01-30 05:39:49,75,M,99.630556,...,1,0,0,1,0,0,0,0,0,0
7,19448158,28451027,30008046,8,96,2154-01-30 05:39:49,2154-01-30 17:39:49,75,M,99.630556,...,1,0,0,1,1,0,0,0,0,0
8,19448158,28451027,30008046,9,108,2154-01-30 17:39:49,2154-01-31 05:39:49,75,M,99.630556,...,1,0,0,1,0,0,0,0,0,0
9,11578593,22558545,30010076,1,12,2139-07-08 07:17:00,2139-07-08 19:17:00,74,M,5.839167,...,0,0,0,0,0,0,0,0,0,0


## 포함/제외 기준 적용

In [ ]:
def _cohort_window_counts(cohort_df):
    los_hours = cohort_df['icu_los_hours'].fillna(0).clip(lower=0)
    total_windows = np.ceil(los_hours / 12).astype('int64')
    candidate_windows = (np.floor((los_hours - 24).clip(lower=0) / 12).astype('int64') - 4).clip(lower=0)
    return total_windows, candidate_windows

def _cohort_counts(step, cohort_df):
    total_windows, candidate_windows = _cohort_window_counts(cohort_df)
    return {
        'step': step,
        'n_subjects': int(cohort_df['subject_id'].nunique()),
        'n_hadm': int(cohort_df['hadm_id'].nunique()),
        'n_stays': int(cohort_df['stay_id'].nunique()),
        'total_12h_windows': int(total_windows.sum()),
        'candidate_12h_windows_excl_first48_last24': int(candidate_windows.sum()),
    }

adm_for_cohort = adm_pat_icu.copy()
for col in ['intime', 'outtime']:
    adm_for_cohort[col] = pd.to_datetime(adm_for_cohort[col], errors='coerce')
adm_for_cohort['icu_los_hours'] = (adm_for_cohort['outtime'] - adm_for_cohort['intime']).dt.total_seconds() / 3600

cohort_flow_rows = []
cohort = adm_for_cohort.copy()
cohort_flow_rows.append(_cohort_counts('0. All ICU stays from extraction', cohort))

cohort = cohort[cohort['icu_los_hours'] >= 72].copy()
cohort_flow_rows.append(_cohort_counts('1. ICU LOS >= 72 hours', cohort))

delirium_assessments_after_72h = (
    timeseries
    .loc[timeseries['feature_name'].eq('delirium') & timeseries['charttime'].notna(), ['stay_id', 'charttime']]
    .merge(adm_for_cohort[['stay_id', 'intime']], on='stay_id', how='left')
)
delirium_assessments_after_72h['hours_since_icu_admit'] = (
    delirium_assessments_after_72h['charttime'] - delirium_assessments_after_72h['intime']
).dt.total_seconds() / 3600
stays_with_delirium_assessment_after_72h = set(
    delirium_assessments_after_72h
    .loc[delirium_assessments_after_72h['hours_since_icu_admit'] > 72, 'stay_id']
)
cohort = cohort[cohort['stay_id'].isin(stays_with_delirium_assessment_after_72h)].copy()
cohort_flow_rows.append(_cohort_counts('2. >=1 delirium assessment after 72 hours', cohort))

cohort_final = cohort.copy()
cohort_final['ever_delirium'] = cohort_final['subject_id'].map(subject_ever_delirium).fillna(0).astype('int8')
cohort_stay_ids = set(cohort_final['stay_id'])
timeseries_cohort = timeseries[timeseries['stay_id'].isin(cohort_stay_ids)].copy()
wide_timeseries_cohort = wide_timeseries[wide_timeseries['stay_id'].isin(cohort_stay_ids)].copy()
wide_timeseries_12h_bin_cohort = wide_timeseries_12h_bin[wide_timeseries_12h_bin['stay_id'].isin(cohort_stay_ids)].copy()

cohort_flow = pd.DataFrame(cohort_flow_rows)
cohort_flow['stays_removed_from_previous'] = cohort_flow['n_stays'].shift(1).fillna(cohort_flow['n_stays']).astype(int) - cohort_flow['n_stays'].astype(int)
cohort_flow['stays_pct_of_initial'] = cohort_flow['n_stays'] / cohort_flow.loc[0, 'n_stays'] * 100

# Cohort-filtered outputs
timeseries_cohort.to_csv(OUTPUT_DIR / 'events_12h_long.csv', index=False)
wide_timeseries_cohort.to_csv(OUTPUT_DIR / 'events_12h_wide_by_charttime.csv', index=False)
wide_timeseries_12h_bin_cohort.to_csv(OUTPUT_DIR / 'events_12h_binned.csv', index=False)
cohort_final.to_csv(OUTPUT_DIR / 'cohort_final.csv', index=False)
cohort_flow.to_csv(OUTPUT_DIR / 'cohort_attrition.csv', index=False)

print(f"Cohort long-format events: {timeseries_cohort.shape}")
print(f"Saved: events_12h_long.csv ({len(timeseries_cohort):,} rows, {timeseries_cohort['stay_id'].nunique()} stays)")
print(f"Saved: events_12h_wide_by_charttime.csv ({len(wide_timeseries_cohort):,} rows, {wide_timeseries_cohort['stay_id'].nunique()} stays)")
print(f"Saved: events_12h_binned.csv ({len(wide_timeseries_12h_bin_cohort):,} rows, {wide_timeseries_12h_bin_cohort['stay_id'].nunique()} stays)")
print(f"Saved: cohort_final.csv ({len(cohort_final):,} stays)")
print(f"Saved: cohort_attrition.csv ({len(cohort_flow):,} rows)")
display(cohort_flow)
